# Does the cue-free form of the fixed-shift hypothesis hold on a fine-tune with no trigger?

*E36, `experiments/regating_spread`, 2026-10-26. Record: `reports/analyses/2026-10-26_analysis_regating-spread.md`.
Follows E35, `notebooks/E35_regating_cuefree.ipynb`. Exploration, provisional.*

## The answer

Yes, once enough positions are compared. On a fine-tune of lowercased Pile text, each position was given the mean
change of the eight training positions with the nearest original importance values. Fitted on 16,384 positions this
recovered 0.72 of Γ's switching at step 10, then 0.54 and 0.51, under the filed 0.6. Fitted on all 262,144 it recovered
0.89, 0.78 and 0.76, and the same rule on the attention inputs Γ reads about as much. At the same sample size the current token
recovers about a third, a summary of the last few tokens about a half, and position in the row almost nothing.

## What we were trying to test

**The setup.** A fine-tune of all 24 decomposed matrices on the next-token loss at every position, by Adam at rate
3e-5 with batches of eight 512-token rows, each row used once. In the *do-nothing* state the update goes into Δ and the
subcomponents stay as released; Γ is frozen but reads the fine-tuned model, and Δ is left out of the CI-masked model.
Everything is read at every position of 64 held-out rows:

- the *gain*: the fine-tuned model's mean next-token log-probability minus the original's;
- the *switching share* $r$: how much the CI-masked model's mean log-probability rises when Γ reads the fine-tuned model
  instead of the original, divided by the gain. 1 means switching alone reproduces the gain;
- each rule's *recovery*: the rise its predicted importance values give, as a share of the rise Γ's own values give.
  0 is the original model's values everywhere, 1 is Γ's values on the fine-tuned model.

**The rules.** Each gives every position its own original importance values $g_0$ plus a predicted change, clipped to
$[0, 1]$. In the scored run the change is fitted at 16,384 random positions of the first 64 training rows:

- *one change everywhere*, the mean change: the original fixed-shift hypothesis;
- *one change per current token*, the mean change at fitted positions holding the same token, or the overall mean for
  tokens seen fewer than 20 times: the toy tasks' "cue" carried over to running text;
- *the 8 most similar positions*, the mean change of the eight fitted positions with the nearest original values: the
  cue-free form E35 supported;
- *k-means groups* of the original values, at 8 to 512 groups, with random groups as their null;
- *a line per subcomponent*, $a\,g_0 + b$ fitted per subcomponent.

**The two tasks.** The planned task was code-like text: the 5% of Pile rows richest in code punctuation. That
fine-tune did not learn, so a second was run on a style instead: Pile text decoded, lowercased and re-encoded, so
capitals never come. Its training rows are used once each over 300 steps; it is read at steps 10, 100 and 300.
The lowercase run is the scored run; three later checks follow it.

## What we expected, and why

- **Switching carries at least 0.3 of the gain**, low-medium confidence: on toy tasks the model already does it carried
  0.5 to 1.1, but a fine-tune with no trigger might work differently.
- **One change everywhere carries less than half**, because positions differ as much as copy-task keys did.
- **The 8 most similar positions recover at least 0.6, and beat one change everywhere by 0.2**, medium for the
  code task and low for the lowercase task after the code run's 0.23 to 0.34.
- **They do at least as well as one change per current token**, within 0.1.
- **The line per subcomponent beats one change everywhere by 0.1**, from its 0.96 to 1.02 on E35's few-cue tasks.
- **512 groups come within 0.1 of the 8 most similar positions.**
- **The filed decision tree** called the case where the nearest rule, the token rule and the line all miss "all fail".
  It read it as: the change at a position depends on more than its own original values.
- **The later checks** predicted that the nearest rule would keep rising with more fitted positions but stay under 0.7
  at step 300. They also predicted that the attention inputs Γ reads would do as well as the original values.

## What we observed

### How much of the gain does switching carry, and what does that show?

**Reading it.** Most of it, but that alone shows little. On lowercase text the switching share was 1.05 at step 10, 0.90
at step 100 and 0.79 at step 300, while the gain grew from 0.07 to 0.13 nats per token. The code-like fine-tune, which
learned nothing and made held-out text worse, gave a share of about 0.8 as well. So a share near 0.8 is what any small
update of these kinds gives here; it does not by itself show that the fine-tune re-weighted subcomponents to learn.

### Does the change follow the original values, the current token, or neither?

![arms](../experiments/regating_spread/figures/amend1/arms.png)

**Reading it.** The original values predict it best, though only about half of it after step 10. The 8 most similar
positions, green, recover 0.72 at step 10, then 0.54 and 0.51, below the filed 0.6 at both later steps. One change per
current token, blue, recovers 0.24 to 0.32, and one change everywhere, grey, 0.03 to 0.07. The token rule is a weak
baseline: only 90 tokens had enough fitted positions, and every other position got the overall change. A third of the
8 nearest positions hold the position's own token, far above the few percent expected by chance, so the nearest
positions are strongly but not wholly the same token.

### How many groups does the switching need?

![clusters](../experiments/regating_spread/figures/amend1/clusters.png)

**Reading it.** Many. Recovery rises with the number of k-means groups, from 0.12 to 0.16 at 8 groups to 0.44 to
0.64 at 512, and stays below the 8 most similar positions at every step. Already at 32 groups the original values beat
the 90 token groups, 0.32 to 0.45 against 0.24 to 0.32, the fairer comparison of the two. Random groups match one
change everywhere to within 0.002. There is no small set of conditions here.

### Is it one fixed line per subcomponent?

**Reading it.** No. The line recovers 0.39 at step 10, 0.14 at step 100 and −0.01 at step 300, against one change
everywhere's 0.03 to 0.07. On the code-like run it recovered about half of a switching that carried no learning. The
slopes barely move from 1: the median among subcomponents whose values vary is 0.94 to 0.96.

### Amendment 2: is the shortfall sampling, and does the token add anything?

The lowercase fine-tune was regenerated, and the 16,384-position nearest rule reproduced the scored run exactly. An
independent check asked whether the rule misses because it drew on too few positions. It also asked whether restricting
the neighbours to the same token helps, as restricting to the cue did on one toy state.

**Reading it.** Partly sampling, and the token adds nothing. With four times as many fitted positions, 65,536, the
nearest rule recovers 0.80, 0.67 and 0.58 at steps 10, 100 and 300, against 0.72, 0.54 and 0.51. That clears the filed
0.6 at step 100 but not at step 300, where the gain was 0.08 against the filed 0.1, so the sampling prediction fails as
filed while the rule keeps improving with more data. Drawing the eight neighbours only from positions with the same
token changes recovery by 0.02 or less at every step. The token rule itself covers only 38% of held-out positions; the
rest fell back on one change everywhere.

| step | nearest, 16,384 positions | nearest, 65,536 | nearest within token, 16,384 |
|---|---|---|---|
| 10 | 0.715 | 0.801 | 0.699 |
| 100 | 0.543 | 0.671 | 0.534 |
| 300 | 0.505 | 0.581 | 0.507 |

### Amendment 3: where does the sampling curve end?

The lowercase fine-tune was regenerated on an H100, and the 16,384- and 65,536-position rules reproduced Amendment 2
within 0.008 at every step. The same nearest rule was then fitted on all 262,144 positions of 512 training rows.

**Reading it.** Something unexpected, in the hypothesis's favour: the curve kept rising, and with the full sample the
rule recovers most of the switching at every step. With 262,144 positions it recovers 0.89, 0.78 and 0.76 at steps
10, 100 and 300, and 0.90, 0.79 and 0.78 with 32 neighbours. We predicted it would still be under 0.7 at step 300.
Each fourfold increase in positions added about 0.08 to 0.18, so the filed "all fail" at step 300 was a sampling
shortfall, which is how the filed tree reads it. The curve had not flattened, and a nearest rule on any rich summary of
the context might improve the same way; that comparison is Amendment 4. A tenth to a quarter of
the switching remains unexplained, more of it later in training.

| step | 16,384 positions | 65,536 | 262,144 | 262,144, 32 neighbours |
|---|---|---|---|---|
| 10 | 0.719 | 0.808 | 0.886 | 0.902 |
| 100 | 0.542 | 0.673 | 0.776 | 0.789 |
| 300 | 0.508 | 0.584 | 0.762 | 0.776 |

### Amendment 4: is Γ's original-value space special?

An independent check of Amendment 3 asked whether a nearest rule on any rich summary of the context would do as well.
So the same rule was fitted, on the same 262,144 positions, in a second space: each layer's attention input under the
original model. These are the values Γ itself reads for the query, key and value matrices. The token rule and a rule per
position in the row were refitted on the full sample.

![sampling](../experiments/regating_spread/figures/sampling.png)

**Reading it.** The two spaces do about equally well, the token stays far behind, and position carries nothing. The
attention inputs recover 0.87, 0.78 and 0.74 at steps 10, 100 and 300, against 0.89, 0.78 and 0.76 for the original importance
values. With the full sample the token rule covers 71% of positions and recovers 0.33 to 0.42, and the rule per
position in the row 0.04 to 0.07. The nearest rule recovers 0.70 to 0.90 in every part of the row, early positions
included, so it does not rest on the starts of rows; the ends of rows, which Γ also sees as edges, are pooled into the
last part. No held-out position lies inside a 16-token span
that also occurs in the fitted rows. So Γ's original values predict the change no better than Γ's own input from the
original model: one sufficient summary of the context, not a privileged one. The gaps, 0.02, are the size of the gain
from 8 to 32 neighbours, and one seed and one set of held-out rows cannot separate them further.

| step | nearest, original values | nearest, attention inputs | token | position in row |
|---|---|---|---|---|
| 10 | 0.886 | 0.869 | 0.416 | 0.069 |
| 100 | 0.776 | 0.777 | 0.344 | 0.046 |
| 300 | 0.762 | 0.744 | 0.334 | 0.038 |

The check added that both spaces come from the original model. A summary the model did not produce is the test that
would separate "how the model represents the context" from "similar text". The token rule left 29% of positions on the
overall mean, so its deficit mixes too little information with too little coverage.

### Amendment 5: does the model's view of the context matter, or only the text?

*Plan: Amendment 7 of the plan. `experiments/regating_spread/amend5.py`, predictions filed 2026-10-26 19:06
(`results/predictions_amend5.json`, sha256 `aa96a068…`) before the run; results `results/regating_spread_amend5.json`.
RunPod H100, 80 s, $0.85.*

Amendment 4's check pointed out that Γ's values and the attention inputs both come from the original model, so they
cannot tell "positions the original model treats alike" from "positions with similar text". This run fits the same
nearest rule, on the same 262,144 positions, on a summary of the text the model did not produce: each of the current
and 15 preceding tokens, or 3, hashed with its offset to one of 4,096 signed coordinates, weighted towards recent
tokens. As a floor, the 4-token summary had to do at least as well as the current-token rule, which it did at every
step; that shows the text space carries the current token's information, not that it could reach the original values
if the text were enough.

**Reading it.** The model's view predicts more than the last few tokens do. The text summaries recover 0.53, 0.54 and 0.50 of the switching with 16
tokens, and 0.56, 0.52 and 0.49 with 4, at steps 10, 100 and 300. The original importance values recover 0.88, 0.77
and 0.76, 0.23 to 0.35 more at every step. The text rules beat the current token, 0.33 to 0.42, so they carry real
information, and about 69% of their neighbours hold the same token. So positions that end in similar text get partly
alike changes, and positions the original model treats alike get much more alike ones. With the weighting used, 83% of
the 16-token summary rests on its last 4 tokens, so the two windows are nearly the same summary, while the model's
view is computed from the whole preceding row.

| step | nearest, original values | nearest, text of 16 tokens | nearest, text of 4 tokens | token rule (Amendment 4) |
|---|---|---|---|---|
| 10 | 0.881 | 0.534 | 0.563 | 0.416 |
| 100 | 0.774 | 0.542 | 0.522 | 0.344 |
| 300 | 0.761 | 0.503 | 0.491 | 0.334 |

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| A0 | the nearest rule on the original values reproduces Amendment 4 | within 0.02 at every step | within 0.005 | pass |
| C_text | the 4-token text summary is at least as good as the token rule (a floor) | ≥ token − 0.05 at every step | +0.15 to +0.18 | pass |
| T1 | the text summary predicts the change worse than the original values | text16 ≤ g0 − 0.1 at step 300 | −0.26 | pass, as predicted |
| T2 | more preceding text helps | text16 ≥ text4 at step 300 | +0.011, −0.030 at step 10 | pass by the rule, below the 0.014 noise floor |

By the filed decision tree, "T1 holds" reads: the original model's view of the context predicts the change better than
the text, so the restatement "positions the original model treats alike get alike changes" is about the model, not the
text. The independent check narrowed that: the text summary saw in effect about four tokens and the model the whole
prefix, so what is shown is "more than positions sharing their last few tokens". A summary of the whole preceding text
is the fairer null; that is Amendment 6.

## What we learned

In the scored run, with 16,384 fitted positions, the cue-free form held at step 10 and failed at steps 100 and 300 by
the filed rule. Amendments 2 and 3 showed that miss was sampling: with all 262,144 positions the rule recovers 0.76 to
0.89 of the switching at every step. So on running text with no trigger, values predicted from the positions with the most similar original
importance values recover most of the switching effect. Amendment 4 found them predicted about as well, within 0.02, from the
original model's attention inputs, which Γ reads, so Γ's original values are one sufficient summary of the context, not
a privileged one. A summary of the last few tokens, which the model did not produce, recovers about a half, 0.23 to
0.35 less (Amendment 5). So the original model's view of the position predicts the change better than its recent
text; whether a summary of the whole preceding text would close the gap is not yet tested, since the model sees the
whole prefix and the text summary in effect about four tokens. At the same sample size the current token recovers about a third and
position in the row almost nothing; at 16,384 positions the original values also beat a few groups and a fixed rule per
subcomponent. A tenth to a quarter of the switching remains unexplained, more of it later in training. A high switching share is not evidence of learned re-weighting
on its own, since an update that learned nothing gave the same share.

## How compelling is the evidence

- **Controls.** At every step a zero change left the masked model's log-probability exactly where it was, and Γ's own
  values reproduced the full switching to within 0.000005.
- **Nulls.** Random groups land within 0.002 of one change everywhere at every step.
- **Held-out positions.** Every rule is scored on 64 held-out rows from another split, 32,704 positions. The scored
  rules are fitted on rows the fine-tune saw in its first eight steps. Amendment 3's 512 rows include 432 it had not
  seen at step 10.
- **Coverage.** One fine-tune seed, one style, one 67M model. Lowercasing is synthetic in form; the more natural task,
  code-like text, did not learn.
- **Blind spots.** The distance weighs every alive subcomponent equally. The rule had not flattened at 262,144
  positions, so its endpoint is not known. Effects passing between positions through attention are included, not
  separated. No predicted pattern was checked for faithfulness under the adversary.

## Surprises

1. **With all positions the rule cleared the bar.** We predicted it would stay under 0.7 at step 300. It recovered
   0.76, so the scored run's miss was a sampling shortfall.
2. **In the scored run the cue-free form held only early.** It recovered 0.72 at step 10, then 0.54 and 0.51, under the
   filed 0.6.
3. **The line per subcomponent collapsed with training.** It beat one change everywhere by 0.32 at step 10, against
   the filed 0.1. It missed by 0.002 at step 100 and fell to −0.01 by step 300.
4. **The code-like fine-tune did not learn.** Held-out text got worse by 0.05 to 0.07 nats per token; its rows were
   mostly LaTeX and maths. Why is untested: the same recipe learned lowercase text.

## What next

1. **Done as Amendment 3:** with all 262,144 positions the rule reaches 0.76 to 0.89 at every step. Next on this: what
   the remaining tenth to quarter depends on, and whether the curve flattens with yet more positions.
2. **Done as Amendments 4 and 5:** Γ's own input from the original model does as well as its values, and a summary of
   the text alone does a quarter worse. What the remaining tenth to quarter depends on, and a richer text summary as a
   harder null, are new designs and go to the user through the pending card.
3. **A fine-tune of a real behaviour that learns**, or the larger model's post-trained sibling, before any restatement
   is adopted. A pending-decision card in the log holds this choice.

## Scorecard

The lowercase run, filed 2026-10-26 17:39 (`results/predictions_amend1.json`, sha256 `48b37e56…`) before the run,
scored at step 300; steps 10 and 100 read by the same rules.

| id | claim | rule | read at step 300 | verdict at steps 10, 100, 300 |
|---|---|---|---|---|
| C0 | zero change and own values check | as filed | exact | pass, pass, pass |
| G | the fine-tune learned and switching carries some of it | gain ≥ 0.02, r ≥ 0.1 | 0.130, 0.79 | pass, pass, pass |
| P_r | switching carries at least 0.3 of the gain | r ≥ 0.3 | 0.79 | pass, pass, pass |
| P_shared | one change everywhere carries under half | R_shared < 0.5 | 0.034 | pass, pass, pass |
| P_near | the 8 most similar positions recover ≥ 0.6 and beat one change by 0.2 | both | 0.505 | pass, **fail**, **fail** |
| P_token | they come within 0.1 of one change per token or beat it | R_near ≥ R_token − 0.1 | 0.505 vs 0.243 | pass, pass, pass |
| P_fix | the line beats one change by 0.1 | R_fix ≥ R_shared + 0.1 | −0.014 | pass, **fail**, **fail** |
| P_K | 512 groups within 0.1 of the nearest rule | R_clu_512 ≥ R_near − 0.1 | 0.442 vs 0.505 | pass, pass, pass |
| N | random groups match one change | within 0.05 | 0.0007 apart | pass, pass, pass |

By the filed decision tree, step 300 falls in the branch "all fail". The code-like run, filed 17:20
(`results/predictions.json`, sha256 `f93a0727…`), failed G, the check that it learned, at both steps, so every
prediction is unscorable.

Amendments 2 to 4, each filed before its run: `results/predictions_amend2.json` (sha256 `eab5cad2…`),
`predictions_amend3.json` (`af874f67…`), `predictions_amend4.json` (`a3ad2c4a…`).

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| A0 | each amendment reproduces the run before it | within 0.02 at every step | exact; within 0.008; exact | pass |
| S | more positions add at least 0.1 at step 300 | R_near65 ≥ R_near16 + 0.1 | +0.076 | fail, as predicted |
| W | restricting to the token adds at most 0.05 | R_near16 ≥ R_near16_tok − 0.05 | −0.002 | pass, as predicted |
| M | still rising at step 300 | R_near262 ≥ R_near65 + 0.03 | +0.178 | pass, as predicted |
| B | the rule recovers at least 0.7 at every step | R_near262 ≥ 0.7 | 0.886, 0.776, 0.762 | **pass, reversing the filed lean** |
| R1 | the attention inputs Γ reads do as well | R_near_resid ≥ R_near_g0 − 0.05 at step 300 | −0.018 | pass, as predicted |
| R2 | position alone carries little | R_position < 0.3 at every step | 0.04 to 0.07 | pass, as predicted |
| R3 | the token stays at least 0.2 behind | R_token ≤ R_near_g0 − 0.2 at every step | 0.43 to 0.47 behind | pass, as predicted |
| D | no near-duplicate text | ≤ 2% of held-out positions | 0 | pass, as predicted |
| C_text | the 4-token text summary is at least as good as the token rule (a floor) | ≥ token − 0.05 at every step | +0.15 to +0.18 | pass |
| T1 | a text summary predicts worse than the original values | text16 ≤ g0 − 0.1 at step 300 | −0.26 | pass, as predicted |
| T2 | more preceding text helps | text16 ≥ text4 at step 300 | +0.011, −0.030 at step 10 | pass by the rule, below the 0.014 noise floor |

## Not run

Seeds beyond 0, and any second set of held-out rows; more than 262,144 fitted positions; numbers of neighbours other
than 8 and 32; groups and the line refitted on 262,144 positions; text summaries other than one hashing with windows of 4 and 16 tokens; residual summaries other than the attention
inputs; any distance but Euclidean; the rules' faithfulness under the adversary; a domain fine-tune that learns; SGD or LoRA
updates; a larger model.

## Provenance

- Code-like run: `experiments/regating_spread/results/regating_spread.json` (sha256 `1b45dcb7…`), log
  `results/large/full.log`; RunPod 4090, run 77 s, on a pod shared with E35's Amendment 1, $0.28 for the pod.
- Lowercase run: `results/regating_spread_amend1.json` (sha256 `dc631b3d…`), log `results/large/amend1.log`; RunPod
  4090, run 90 s, $0.12.
- Amendments 2, 3 and 4: `results/regating_spread_amend{2,3,4}.json`; RunPod 4090, H100 and H100 NVL; 58, 85 and 98 s;
  $0.09, $0.30 and $0.44.
- Code `experiments/regating_spread/run.py` and `amend1.py` to `amend4.py`, shared steps in `src/commute/spread.py` and
  `src/commute/cuefree.py`; plan `reports/plans/2026-10-26_plan_regating-without-cues.md`. Figures
  `figures/amend1/{arms,clusters}.png` from `plot.py`, `figures/sampling.png` from `plot_sampling.py`.